# Celda 1 — Montar Drive e imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE_PATH   = '/content/drive/MyDrive/project_factoria/waste_classifier'
UNIFIED     = os.path.join(BASE_PATH, 'datasets/unified')
METADATA    = os.path.join(BASE_PATH, 'datasets/metadata')
CHECKPOINTS = os.path.join(BASE_PATH, 'checkpoints')
LOGS        = os.path.join(BASE_PATH, 'logs')
OUTPUTS     = os.path.join(BASE_PATH, 'outputs')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("STOP — activa la GPU antes de continuar")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda
GPU: Tesla T4


# Celda 2 — DataLoaders con augmentation reforzada para metal/plastico/vidrio


In [3]:
with open(os.path.join(METADATA, 'config.json')) as f:
    config = json.load(f)
with open(os.path.join(METADATA, 'class_weights.json')) as f:
    weights_dict = json.load(f)

IMAGENET_MEAN = config['imagenet_mean']
IMAGENET_STD  = config['imagenet_std']

# Augmentation más agresiva — especialmente en color y brillo
# para que metal/plastico/vidrio no se distingan solo por superficie
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3,
        hue=0.08
    ),
    transforms.RandomGrayscale(p=0.05),  # ocasionalmente quita el color
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

dataset_train = datasets.ImageFolder(os.path.join(UNIFIED, 'train'), transform=transform_train)
dataset_val   = datasets.ImageFolder(os.path.join(UNIFIED, 'val'),   transform=transform_val)

loader_train = DataLoader(dataset_train, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
loader_val   = DataLoader(dataset_val,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_weights = torch.tensor(
    [weights_dict[c] for c in dataset_train.classes],
    dtype=torch.float
).to(device)

print("Clases:", dataset_train.classes)
print(f"Batches train: {len(loader_train)} | val: {len(loader_val)}")

Clases: ['carton', 'metal', 'no_reciclable', 'organico', 'papel', 'plastico', 'vidrio']
Batches train: 571 | val: 72


# Celda 3 — Cargar best_model y descongelar las últimas capas del backbone


In [4]:
# Reconstruir la arquitectura exacta del entrenamiento anterior
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, len(dataset_train.classes))
)

# Cargar los pesos del mejor modelo anterior
checkpoint = torch.load(
    os.path.join(CHECKPOINTS, 'best_model.pth'),
    map_location=device
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

print(f"Modelo cargado desde epoch {checkpoint['epoch']} "
      f"(val_acc: {checkpoint['val_acc']*100:.2f}%)")

# --- Descongelar selectivamente ---
# EfficientNet-B0 tiene bloques MBConv numerados de 0 a 8
# Descongelamos solo los últimos 3 bloques (6, 7, 8) + el head conv final
# Los primeros bloques detectan features genericos (bordes, texturas) — los dejamos congelados
# Los últimos bloques detectan features de alto nivel (formas, materiales) — esos necesitan ajuste

for name, param in model.named_parameters():
    param.requires_grad = False  # primero todo congelado

for name, param in model.named_parameters():
    if any(layer in name for layer in [
        'features.6', 'features.7', 'features.8',  # últimos 3 bloques MBConv
        'classifier'                                  # clasificador
    ]):
        param.requires_grad = True

# Verificar
total     = sum(p.numel() for p in model.parameters())
entrena   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParametros totales:     {total:,}")
print(f"Parametros entrenables: {entrena:,}  ({100*entrena/total:.1f}%)")
print("\nCapas descongeladas:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

Modelo cargado desde epoch 15 (val_acc: 86.95%)

Parametros totales:     4,016,515
Parametros entrenables: 3,164,707  (78.8%)

Capas descongeladas:
  features.6.0.block.0.0.weight
  features.6.0.block.0.1.weight
  features.6.0.block.0.1.bias
  features.6.0.block.1.0.weight
  features.6.0.block.1.1.weight
  features.6.0.block.1.1.bias
  features.6.0.block.2.fc1.weight
  features.6.0.block.2.fc1.bias
  features.6.0.block.2.fc2.weight
  features.6.0.block.2.fc2.bias
  features.6.0.block.3.0.weight
  features.6.0.block.3.1.weight
  features.6.0.block.3.1.bias
  features.6.1.block.0.0.weight
  features.6.1.block.0.1.weight
  features.6.1.block.0.1.bias
  features.6.1.block.1.0.weight
  features.6.1.block.1.1.weight
  features.6.1.block.1.1.bias
  features.6.1.block.2.fc1.weight
  features.6.1.block.2.fc1.bias
  features.6.1.block.2.fc2.weight
  features.6.1.block.2.fc2.bias
  features.6.1.block.3.0.weight
  features.6.1.block.3.1.weight
  features.6.1.block.3.1.bias
  features.6.2.block.0.0

# Celda 4 — Optimizer con learning rates diferenciales


In [5]:
# Learning rates diferenciados — critico en fine-tuning
# El backbone ya tiene buenos pesos: LR muy pequeño para no destruirlos
# El clasificador puede cambiar más: LR algo mayor

LR_BACKBONE    = 1e-5   # 100x más pequeño que en el entrenamiento inicial
LR_CLASSIFIER  = 1e-4   # 10x más pequeño que en el entrenamiento inicial
NUM_EPOCHS_FT  = 10

# Separar parámetros del backbone de los del clasificador
backbone_params    = [p for n, p in model.named_parameters()
                      if p.requires_grad and 'classifier' not in n]
classifier_params  = [p for n, p in model.named_parameters()
                      if p.requires_grad and 'classifier' in n]

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW([
    {'params': backbone_params,   'lr': LR_BACKBONE},
    {'params': classifier_params, 'lr': LR_CLASSIFIER}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS_FT, eta_min=1e-7
)

print(f"LR backbone descongelado: {LR_BACKBONE}")
print(f"LR clasificador:          {LR_CLASSIFIER}")
print(f"Scheduler: CosineAnnealingLR — decae suavemente hasta 1e-7")
print(f"Epochs de fine-tuning:    {NUM_EPOCHS_FT}")
print(f"\nParametros en backbone group:    {sum(p.numel() for p in backbone_params):,}")
print(f"Parametros en classifier group:  {sum(p.numel() for p in classifier_params):,}")

LR backbone descongelado: 1e-05
LR clasificador:          0.0001
Scheduler: CosineAnnealingLR — decae suavemente hasta 1e-7
Epochs de fine-tuning:    10

Parametros en backbone group:    3,155,740
Parametros en classifier group:  8,967


# Celda 5 — Funciones de train y val (idénticas al notebook anterior)


In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

print("Funciones definidas")

Funciones definidas


# Celda 6 — Loop de fine-tuning


In [ ]:
# Cargar historial anterior para que las curvas sean continuas
history_prev = checkpoint.get('history', {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[]})

history_ft = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = checkpoint['val_acc']  # partimos del 86.95%
best_epoch   = 0

print(f"Fine-tuning — {NUM_EPOCHS_FT} epochs")
print(f"Baseline a superar: {best_val_acc*100:.2f}%")
print(f"{'Epoch':<7} {'T_loss':<10} {'T_acc':<10} {'V_loss':<10} {'V_acc':<10} {'LR_bb':<12} {'Estado'}")
print("-" * 72)

for epoch in range(1, NUM_EPOCHS_FT + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, loader_train, criterion, optimizer, device)
    val_loss,   val_acc   = val_epoch(model, loader_val, criterion, device)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['val_loss'].append(val_loss)
    history_ft['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    estado  = ''

    torch.save({
        'epoch':               epoch,
        'model_state_dict':    model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
        'val_acc':             val_acc,
        'val_loss':            val_loss,
        'classes':             dataset_train.classes,
        'history':             history_ft,
        'history_prev':        history_prev
    }, os.path.join(CHECKPOINTS, 'last_finetune.pth'))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        estado       = '★ mejor'
        torch.save({
            'epoch':               epoch,
            'model_state_dict':    model.state_dict(),
            'optimizer_state_dict':optimizer.state_dict(),
            'val_acc':             val_acc,
            'val_loss':            val_loss,
            'classes':             dataset_train.classes,
            'history':             history_ft,
            'history_prev':        history_prev
        }, os.path.join(CHECKPOINTS, 'best_model_ft.pth'))

    print(f"{epoch:<7} {train_loss:<10.4f} {train_acc*100:<10.2f} "
          f"{val_loss:<10.4f} {val_acc*100:<10.2f} {current_lr:<12.2e} {estado}")

if best_epoch > 0:
    print(f"\nMejora conseguida: {best_val_acc*100:.2f}%  (epoch {best_epoch})")
else:
    print(f"\nNo se supero el baseline de {checkpoint['val_acc']*100:.2f}%")
    print("El best_model.pth original sigue siendo el mejor modelo")

Fine-tuning — 10 epochs
Baseline a superar: 86.95%
Epoch   T_loss     T_acc      V_loss     V_acc      LR_bb        Estado
------------------------------------------------------------------------
1       0.6085     78.47      0.3669     88.01      9.76e-06     ★ mejor
2       0.5580     79.57      0.3846     87.43      9.05e-06     
3       0.5154     81.11      0.3789     87.13      7.96e-06     
4       0.4965     82.43      0.3203     88.97      6.58e-06     ★ mejor
5       0.4637     83.00      0.3157     89.63      5.05e-06     ★ mejor
6       0.4557     83.57      0.3053     90.33      3.52e-06     ★ mejor
7       0.4358     83.93      0.3005     90.07      2.14e-06     
8       0.4335     84.18      0.2952     90.03      1.05e-06     
9       0.4300     84.12      0.3085     89.32      3.42e-07     
10      0.4302     84.25      0.3010     89.98      1.00e-07     

Mejora conseguida: 90.33%  (epoch 6)


# Celda 7 — Curvas comparativas (entrenamiento original vs fine-tuning)


In [ ]:
prev_acc = history_prev.get('val_acc', [])
ft_acc   = history_ft['val_acc']
combined = prev_acc + ft_acc

epochs_prev = list(range(1, len(prev_acc) + 1))
epochs_ft   = list(range(len(prev_acc) + 1, len(combined) + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Entrenamiento completo — fase 1 (frozen) + fase 2 (fine-tuning)', fontweight='bold')

# Loss
axes[0].plot(epochs_prev, history_prev.get('val_loss', []), 'r--o', markersize=3, alpha=0.5, label='Val loss fase 1')
axes[0].plot(epochs_ft,   history_ft['val_loss'],           'r-o',  markersize=4, label='Val loss fine-tuning')
axes[0].axvline(len(prev_acc) + 0.5, color='gray', linestyle=':', alpha=0.7, label='Inicio fine-tuning')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Val Loss')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_prev, [a*100 for a in history_prev.get('val_acc', [])], 'b--o', markersize=3, alpha=0.5, label='Val acc fase 1')
axes[1].plot(epochs_ft,   [a*100 for a in history_ft['val_acc']],           'b-o',  markersize=4, label='Val acc fine-tuning')
axes[1].axvline(len(prev_acc) + 0.5, color='gray', linestyle=':', alpha=0.7, label='Inicio fine-tuning')
axes[1].axhline(86.95, color='orange', linestyle='--', alpha=0.7, label='Baseline 86.95%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Val Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS, 'finetuning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: finetuning_curves.png")